# Grouped / Scaffold Validation for AMD/PAMD Lipid Library

This notebook keeps the original feature construction idea, but uses group-aware validation to reduce structural leakage from the combinatorial head-tail matrix.


In [ ]:
import os
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors
from rdkit.Chem.Scaffolds import MurckoScaffold

from sklearn.base import clone
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    roc_auc_score,
    average_precision_score,
)
from sklearn.model_selection import (
    GroupKFold,
    GroupShuffleSplit,
    GridSearchCV,
    RandomizedSearchCV,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from xgboost import XGBClassifier
import lightgbm as lgb

warnings.filterwarnings("ignore")
plt.rcParams["font.sans-serif"] = ["Arial"]
plt.rcParams["axes.unicode_minus"] = False


# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------
# This notebook can be run from any location because data paths are fixed here.
BASE_DIR = os.path.abspath(os.environ.get("AMD_PAMD_DATA_DIR", os.path.join(os.getcwd(), "data")))
OUTPUT_DIR = os.path.abspath(os.environ.get("AMD_PAMD_OUTPUT_DIR", os.path.join(os.getcwd(), "outputs")))
os.makedirs(OUTPUT_DIR, exist_ok=True)
HEAD_DIR = os.path.join(BASE_DIR, "Sequential structure of the heat map", "head")
TAIL_DIR = os.path.join(BASE_DIR, "Sequential structure of the heat map", "tail")
DATASET_PATH = os.path.join(BASE_DIR, "Dataset.xlsx")

# Reviewer-oriented validation mode:
#   "head"          : all samples with the same head group stay in the same split
#   "tail"          : all samples with the same tail group stay in the same split
#   "head_scaffold" : all samples sharing the same head Murcko scaffold stay in the same split
#   "tail_scaffold" : all samples sharing the same tail Murcko scaffold stay in the same split
#   "pair_scaffold" : all samples sharing the same head+tail scaffold pair stay in the same split
GROUP_MODE = "head"

TEST_SIZE = 0.20
RANDOM_STATE = 42
POSITIVE_THRESHOLD = 2000


def natural_key(text):
    """Sort filenames reproducibly by embedded numbers."""
    return [int(tok) if tok.isdigit() else tok.lower() for tok in re.split(r"(\d+)", text)]


def read_sdf_folder_custom(folder_path):
    """Read SDF files and return molecules plus filenames in a reproducible order."""
    molecules = []
    filenames = []
    for filename in sorted(os.listdir(folder_path), key=natural_key):
        if filename.lower().endswith(".sdf"):
            filepath = os.path.join(folder_path, filename)
            with open(filepath, "r", encoding="utf-8") as f:
                molblock = f.read()
            mol = Chem.MolFromMolBlock(molblock)
            if mol is not None:
                molecules.append(mol)
                filenames.append(filename)
    return molecules, filenames


all_desc_names = [desc[0] for desc in Descriptors._descList]


def calc_208_descriptors(mol):
    """Calculate RDKit descriptors."""
    values = []
    for name in all_desc_names:
        func = getattr(Descriptors, name)
        try:
            values.append(func(mol))
        except Exception:
            values.append(0.0)
    return values


def calc_features(mol):
    """Morgan fingerprint + RDKit descriptors, following the original code."""
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=2048)
    return list(fp) + calc_208_descriptors(mol)


def murcko_scaffold_id(mol, prefix, fallback_id):
    """
    Return a scaffold identifier.

    Lipid tails may have no ring scaffold, so an empty Murcko scaffold is
    replaced by the molecule's own canonical SMILES. This avoids putting all
    acyclic tails into one artificial group.
    """
    try:
        scaffold = MurckoScaffold.GetScaffoldForMol(mol)
        scaffold_smiles = Chem.MolToSmiles(scaffold, isomericSmiles=True)
    except Exception:
        scaffold_smiles = ""

    if not scaffold_smiles:
        try:
            scaffold_smiles = Chem.MolToSmiles(mol, isomericSmiles=True)
        except Exception:
            scaffold_smiles = fallback_id

    return f"{prefix}:{scaffold_smiles}"


def make_grouped_dataset():
    head_mols, head_files = read_sdf_folder_custom(HEAD_DIR)
    tail_mols, tail_files = read_sdf_folder_custom(TAIL_DIR)

    print(f"Number of head molecules: {len(head_mols)}")
    print(f"Number of tail molecules: {len(tail_mols)}")

    head_index = [f"A{i + 1}" for i in range(len(head_mols))]
    tail_index = [f"O{i + 1}" for i in range(len(tail_mols))]

    head_features = [calc_features(m) for m in head_mols]
    tail_features = [calc_features(m) for m in tail_mols]
    head_scaffolds = [murcko_scaffold_id(m, "H", idx) for m, idx in zip(head_mols, head_index)]
    tail_scaffolds = [murcko_scaffold_id(m, "T", idx) for m, idx in zip(tail_mols, tail_index)]

    head_df = pd.DataFrame(head_features, index=head_index)
    tail_df = pd.DataFrame(tail_features, index=tail_index)

    head_meta = pd.DataFrame(
        {"head": head_index, "head_file": head_files, "head_scaffold": head_scaffolds}
    ).set_index("head")
    tail_meta = pd.DataFrame(
        {"tail": tail_index, "tail_file": tail_files, "tail_scaffold": tail_scaffolds}
    ).set_index("tail")

    # Save mapping so the split is transparent in supplementary materials.
    mapping_path = os.path.join(OUTPUT_DIR, "head_tail_index_mapping_group_validation_v1.csv")
    pd.concat(
        [
            head_meta.reset_index().assign(type="head").rename(columns={"head": "id", "head_file": "file", "head_scaffold": "scaffold"}),
            tail_meta.reset_index().assign(type="tail").rename(columns={"tail": "id", "tail_file": "file", "tail_scaffold": "scaffold"}),
        ],
        ignore_index=True,
    )[["type", "id", "file", "scaffold"]].to_csv(mapping_path, index=False, encoding="utf-8-sig")
    print(f"Saved head/tail mapping to: {mapping_path}")

    df = pd.read_excel(DATASET_PATH, index_col=0)

    X, y, meta_rows = [], [], []
    for h in df.index:
        for t in df.columns:
            h_id = str(h)
            t_id = str(t)
            if h_id in head_df.index and t_id in tail_df.index:
                X.append(list(head_df.loc[h_id]) + list(tail_df.loc[t_id]))
                label = 1 if df.loc[h, t] >= POSITIVE_THRESHOLD else 0
                y.append(label)
                meta_rows.append(
                    {
                        "head": h_id,
                        "tail": t_id,
                        "head_file": head_meta.loc[h_id, "head_file"],
                        "tail_file": tail_meta.loc[t_id, "tail_file"],
                        "head_scaffold": head_meta.loc[h_id, "head_scaffold"],
                        "tail_scaffold": tail_meta.loc[t_id, "tail_scaffold"],
                        "pair_scaffold": head_meta.loc[h_id, "head_scaffold"] + "||" + tail_meta.loc[t_id, "tail_scaffold"],
                    }
                )

    X = pd.DataFrame(X)
    y = pd.Series(y, name="label")
    meta = pd.DataFrame(meta_rows)

    print(f"Sample size: {len(y)}")
    print(f"Class distribution:\n{y.value_counts()}")
    print(f"Available group counts:")
    for col in ["head", "tail", "head_scaffold", "tail_scaffold", "pair_scaffold"]:
        print(f"  {col}: {meta[col].nunique()}")

    return X, y, meta


X, y, sample_meta = make_grouped_dataset()


In [ ]:
def choose_group_holdout_split(X, y, groups, test_size=0.2, random_state=42, n_attempts=500):
    """
    GroupShuffleSplit can create a test set with severe class imbalance when
    the number of groups is small. Try many random states and choose a split
    whose test size and positive rate are close to the full dataset.
    """
    y_array = np.asarray(y)
    full_pos_rate = y_array.mean()
    best = None

    for seed in range(random_state, random_state + n_attempts):
        splitter = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
        train_idx, test_idx = next(splitter.split(X, y, groups=groups))
        y_train_fold = y_array[train_idx]
        y_test_fold = y_array[test_idx]

        # Both train and test must contain positive and negative samples.
        if len(np.unique(y_train_fold)) < 2 or len(np.unique(y_test_fold)) < 2:
            continue

        observed_test_size = len(test_idx) / len(y_array)
        test_pos_rate = y_test_fold.mean()
        score = abs(observed_test_size - test_size) + abs(test_pos_rate - full_pos_rate)

        if best is None or score < best[0]:
            best = (score, train_idx, test_idx, seed)

    if best is None:
        raise ValueError("No valid grouped split found. Try a different GROUP_MODE or TEST_SIZE.")

    return best[1], best[2], best[3]


groups = sample_meta[GROUP_MODE].astype(str)
train_idx, test_idx, split_seed = choose_group_holdout_split(
    X, y, groups, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

X_train = X.iloc[train_idx].reset_index(drop=True)
X_test = X.iloc[test_idx].reset_index(drop=True)
y_train = y.iloc[train_idx].reset_index(drop=True)
y_test = y.iloc[test_idx].reset_index(drop=True)
meta_train = sample_meta.iloc[train_idx].reset_index(drop=True)
meta_test = sample_meta.iloc[test_idx].reset_index(drop=True)
groups_train = meta_train[GROUP_MODE].astype(str)
groups_test = meta_test[GROUP_MODE].astype(str)

print(f"Validation mode: {GROUP_MODE}")
print(f"Selected split seed: {split_seed}")
print(f"Training set: {X_train.shape}, Test set: {X_test.shape}")
print(f"Training class distribution:\n{y_train.value_counts()}")
print(f"Test class distribution:\n{y_test.value_counts()}")
print(f"Train groups: {groups_train.nunique()}, Test groups: {groups_test.nunique()}")
print(f"Shared groups between train and test: {len(set(groups_train) & set(groups_test))}")

split_table = pd.concat(
    [
        meta_train.assign(split="train", label=y_train.values),
        meta_test.assign(split="test", label=y_test.values),
    ],
    ignore_index=True,
)
split_path = os.path.join(OUTPUT_DIR, f"group_validation_split_{GROUP_MODE}_v1.csv")
split_table.to_csv(split_path, index=False, encoding="utf-8-sig")
print(f"Saved split table to: {split_path}")

n_inner_splits = min(5, groups_train.nunique())
if n_inner_splits < 2:
    raise ValueError("Too few training groups for grouped cross-validation.")
inner_cv = GroupKFold(n_splits=n_inner_splits)


In [ ]:
def evaluate_on_group_holdout(model_name, estimator, X_test, y_test, threshold=0.5):
    prob = estimator.predict_proba(X_test)[:, 1]
    pred = (prob >= threshold).astype(int)

    metrics = {
        "model": model_name,
        "group_mode": GROUP_MODE,
        "threshold": threshold,
        "accuracy": accuracy_score(y_test, pred),
        "f1": f1_score(y_test, pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, prob),
        "pr_auc": average_precision_score(y_test, prob),
    }

    print("\n" + "=" * 24 + f" {model_name} " + "=" * 24)
    print(f"Threshold: {threshold:.3f}")
    print(f"Accuracy: {metrics['accuracy']:.4f}")
    print(f"F1: {metrics['f1']:.4f}")
    print(f"ROC AUC: {metrics['roc_auc']:.4f}")
    print(f"PR AUC: {metrics['pr_auc']:.4f}")
    print("Confusion matrix:")
    print(confusion_matrix(y_test, pred))
    print("Classification report:")
    print(classification_report(y_test, pred, zero_division=0))
    return metrics


results = []


In [ ]:
# ---------------------------------------------------------------------
# Model 1: Random Forest
# ---------------------------------------------------------------------
rf_pipe = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("pca", PCA(random_state=RANDOM_STATE)),
        ("clf", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)),
    ]
)

rf_param_grid = {
    "pca__n_components": [30, 50, 80],
    "clf__n_estimators": [100, 300],
    "clf__max_depth": [3, 5, 7, None],
    "clf__min_samples_split": [5, 10, 20],
    "clf__min_samples_leaf": [2, 5, 10],
    "clf__max_features": ["sqrt"],
    "clf__class_weight": ["balanced"],
}

rf_search = GridSearchCV(
    rf_pipe,
    rf_param_grid,
    cv=inner_cv,
    scoring="f1",
    n_jobs=-1,
    refit=True,
)
rf_search.fit(X_train, y_train, groups=groups_train)
print("Best Random Forest parameters:", rf_search.best_params_)
results.append(evaluate_on_group_holdout("Random Forest", rf_search.best_estimator_, X_test, y_test))


In [ ]:
# ---------------------------------------------------------------------
# Model 2: XGBoost
# ---------------------------------------------------------------------
neg_count = np.sum(y_train == 0)
pos_count = np.sum(y_train == 1)
base_scale_pos_weight = neg_count / max(pos_count, 1)

xgb_pipe = Pipeline(
    [
        (
            "selector",
            SelectFromModel(
                RandomForestClassifier(
                    n_estimators=300,
                    class_weight="balanced",
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                ),
                max_features=300,
                threshold="median",
            ),
        ),
        (
            "clf",
            XGBClassifier(
                objective="binary:logistic",
                eval_metric="logloss",
                random_state=RANDOM_STATE,
                n_jobs=-1,
                tree_method="hist",
            ),
        ),
    ]
)

xgb_param_dist = {
    "selector__max_features": [150, 300, 500],
    "clf__n_estimators": [300, 500, 700],
    "clf__max_depth": [2, 3, 4],
    "clf__learning_rate": [0.005, 0.01, 0.03, 0.05],
    "clf__subsample": [0.7, 0.8, 0.9, 1.0],
    "clf__colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "clf__gamma": [0, 0.05, 0.1, 0.3],
    "clf__min_child_weight": [1, 3, 5],
    "clf__reg_alpha": [0, 0.01, 0.05, 0.1],
    "clf__reg_lambda": [0.5, 1, 2, 5],
    "clf__scale_pos_weight": [
        base_scale_pos_weight * 0.7,
        base_scale_pos_weight,
        base_scale_pos_weight * 1.2,
        base_scale_pos_weight * 1.5,
    ],
}

xgb_search = RandomizedSearchCV(
    xgb_pipe,
    xgb_param_dist,
    n_iter=80,
    scoring="f1",
    cv=inner_cv,
    verbose=0,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    refit=True,
)
xgb_search.fit(X_train, y_train, groups=groups_train)
print("Best XGBoost parameters:", xgb_search.best_params_)
results.append(evaluate_on_group_holdout("XGBoost", xgb_search.best_estimator_, X_test, y_test))


In [ ]:
# ---------------------------------------------------------------------
# Model 3: LightGBM
# ---------------------------------------------------------------------
lgbm_pipe = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("pca", PCA(random_state=RANDOM_STATE)),
        ("clf", lgb.LGBMClassifier(objective="binary", random_state=RANDOM_STATE, verbosity=-1)),
    ]
)

lgbm_param_dist = {
    "pca__n_components": [30, 50, 80],
    "clf__n_estimators": [100, 200, 300, 500],
    "clf__max_depth": [4, 6, 8, -1],
    "clf__learning_rate": [0.01, 0.03, 0.05, 0.1],
    "clf__num_leaves": [15, 31, 50],
    "clf__subsample": [0.7, 0.8, 1.0],
    "clf__colsample_bytree": [0.7, 0.8, 1.0],
    "clf__reg_alpha": [0, 0.1, 0.5],
    "clf__reg_lambda": [0, 0.1, 0.5],
    "clf__scale_pos_weight": [
        base_scale_pos_weight * 0.8,
        base_scale_pos_weight,
        base_scale_pos_weight * 1.2,
    ],
}

lgbm_search = RandomizedSearchCV(
    lgbm_pipe,
    lgbm_param_dist,
    n_iter=50,
    scoring="f1",
    cv=inner_cv,
    verbose=0,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    refit=True,
)
lgbm_search.fit(X_train, y_train, groups=groups_train)
print("Best LightGBM parameters:", lgbm_search.best_params_)
results.append(evaluate_on_group_holdout("LightGBM", lgbm_search.best_estimator_, X_test, y_test))


In [ ]:
# ---------------------------------------------------------------------
# Model 4: Logistic Regression
# ---------------------------------------------------------------------
from sklearn.linear_model import LogisticRegression

lr_pipe = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("pca", PCA(random_state=RANDOM_STATE)),
        (
            "clf",
            LogisticRegression(
                penalty="elasticnet",
                solver="saga",
                max_iter=5000,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

lr_param_grid = {
    "pca__n_components": [30, 50, 80, 100],
    "clf__C": [0.1, 1, 10, 100],
    "clf__l1_ratio": [0.1, 0.5, 0.9],
    "clf__class_weight": ["balanced", None],
}

lr_search = GridSearchCV(
    lr_pipe,
    lr_param_grid,
    cv=inner_cv,
    scoring="f1",
    n_jobs=-1,
    refit=True,
)
lr_search.fit(X_train, y_train, groups=groups_train)
print("Best Logistic Regression parameters:", lr_search.best_params_)
results.append(evaluate_on_group_holdout("Logistic Regression", lr_search.best_estimator_, X_test, y_test))


In [ ]:
results_df = pd.DataFrame(results)
print("\nGrouped/scaffold validation summary:")
print(results_df)

results_path = os.path.join(OUTPUT_DIR, f"group_validation_results_{GROUP_MODE}_v1.csv")
results_df.to_csv(results_path, index=False, encoding="utf-8-sig")
print(f"Saved results to: {results_path}")
